# IMDB Movie Review Sentiment Analysis using GRU

In [ ]:
# ============================================================
# IMDB Movie Review Sentiment Analysis using GRU
# ============================================================

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# Load Dataset
# -----------------------------
vocab_size = 10000
max_length = 200

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(
    num_words=vocab_size
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

print("\nSample Encoded Review:")
print(X_train[0])

print("\nSentiment:")
print("Positive" if y_train[0] else "Negative")

# -----------------------------
# Pad Sequences
# -----------------------------
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

# -----------------------------
# Build GRU Model
# -----------------------------
model = keras.Sequential([

    keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_length
    ),

    keras.layers.GRU(
        128,
        return_sequences=False
    ),

    keras.layers.Dense(
        64,
        activation="relu"
    ),

    keras.layers.Dropout(0.5),

    keras.layers.Dense(
        1,
        activation="sigmoid"
    )

])

# -----------------------------
# Model Summary
# -----------------------------
model.summary()

# -----------------------------
# Compile Model
# -----------------------------
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# -----------------------------
# Train Model
# -----------------------------
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# -----------------------------
# Evaluate Model
# -----------------------------
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test
)

print("\nTest Loss :", test_loss)
print("Test Accuracy :", test_accuracy)

# -----------------------------
# Plot Accuracy
# -----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# -----------------------------
# Plot Loss
# -----------------------------
plt.subplot(1,2,2)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# -----------------------------
# Predict Sentiment
# -----------------------------
predictions = model.predict(X_test)

# -----------------------------
# Display Sample Predictions
# -----------------------------
print("\nSample Predictions")
print("-"*50)

for i in range(10):

    predicted = "Positive" if predictions[i][0] >= 0.5 else "Negative"
    actual = "Positive" if y_test[i] else "Negative"

    print(f"Review {i+1}")
    print(f"Actual    : {actual}")
    print(f"Predicted : {predicted}")
    print("-"*35)

# -----------------------------
# Predict a Single Review
# -----------------------------
index = 100

prediction = model.predict(
    X_test[index].reshape(1, max_length)
)

sentiment = "Positive" if prediction[0][0] >= 0.5 else "Negative"

print("\nSelected Review Prediction")
print("----------------------------")
print("Actual Sentiment    :", "Positive" if y_test[index] else "Negative")
print("Predicted Sentiment :", sentiment)
print("Prediction Score    :", prediction[0][0])